# Train baseline:

In [ ]:
python3 - <<'PY'
import json
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

path = "data/news/event_intelligence/modeling/news_event_modeling_base.parquet"
out = Path("data/news/event_intelligence/modeling/models")
out.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(path)

# Time split
df["news_date"] = pd.to_datetime(df["news_date"])
train = df[df["news_date"] < "2025-01-01"].copy()
valid = df[(df["news_date"] >= "2025-01-01") & (df["news_date"] < "2026-01-01")].copy()
test = df[df["news_date"] >= "2026-01-01"].copy()

target = "target_alpha_pos_5d"

features = [
    "source", "news_category",
    "sector", "subsector", "industry", "subindustry", "listing_board",
    "event_scope", "event_type", "event_side", "impact_channel",
    "materiality_label",
    "materiality_score", "news_intensity_score", "uncertainty_score",
    "novelty_score",
    "bullish_keyword_hits", "bearish_keyword_hits", "uncertainty_keyword_hits",
    "same_event_seen_so_far",
    "daily_ret", "volume_ratio", "bwd_volatility_20d", "drawdown_20d",
    "bwd_ret_1d", "bwd_ret_3d", "bwd_ret_5d", "bwd_ret_7d", "bwd_ret_14d", "bwd_ret_30d",
    "bwd_volume_ratio_1d", "bwd_volume_ratio_3d", "bwd_volume_ratio_5d",
    "bwd_volume_ratio_7d", "bwd_volume_ratio_14d", "bwd_volume_ratio_30d",
]

features = [c for c in features if c in df.columns]
cat_features = [
    c for c in features
    if c in [
        "source", "news_category", "sector", "subsector", "industry", "subindustry",
        "listing_board", "event_scope", "event_type", "event_side",
        "impact_channel", "materiality_label"
    ]
]

for part in [train, valid, test]:
    for c in features:
        if c in cat_features:
            part[c] = part[c].astype("string").fillna("UNKNOWN")
        else:
            part[c] = pd.to_numeric(part[c], errors="coerce").fillna(0)

train_pool = Pool(train[features], train[target], cat_features=cat_features)
valid_pool = Pool(valid[features], valid[target], cat_features=cat_features)
test_pool = Pool(test[features], test[target], cat_features=cat_features)

model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=80,
)

model.fit(train_pool, eval_set=valid_pool)

pred = model.predict_proba(test_pool)[:, 1]
auc = roc_auc_score(test[target], pred)
ap = average_precision_score(test[target], pred)

print("TEST AUC:", auc)
print("TEST AP :", ap)

test_pred_label = (pred >= 0.5).astype(int)
print(classification_report(test[target], test_pred_label, digits=4))

model.save_model(str(out / "news_impact_catboost.cbm"))

metrics = {
    "target": target,
    "features": features,
    "cat_features": cat_features,
    "train_rows": len(train),
    "valid_rows": len(valid),
    "test_rows": len(test),
    "test_auc": float(auc),
    "test_average_precision": float(ap),
}

(out / "news_impact_catboost_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved:", out)
PY

# Train baseline News Risk Model

In [ ]:
python3 - <<'PY'
import json
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

path = "data/news/event_intelligence/modeling/news_event_modeling_base.parquet"
out = Path("data/news/event_intelligence/modeling/models")
out.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(path)

# Time split
df["news_date"] = pd.to_datetime(df["news_date"])
train = df[df["news_date"] < "2025-01-01"].copy()
valid = df[(df["news_date"] >= "2025-01-01") & (df["news_date"] < "2026-01-01")].copy()
test = df[df["news_date"] >= "2026-01-01"].copy()

target = "target_downside_risk_5d"

features = [
    "source", "news_category",
    "sector", "subsector", "industry", "subindustry", "listing_board",
    "event_scope", "event_type", "event_side", "impact_channel",
    "materiality_label",
    "materiality_score", "news_intensity_score", "uncertainty_score",
    "novelty_score",
    "bullish_keyword_hits", "bearish_keyword_hits", "uncertainty_keyword_hits",
    "same_event_seen_so_far",
    "daily_ret", "volume_ratio", "bwd_volatility_20d", "drawdown_20d",
    "bwd_ret_1d", "bwd_ret_3d", "bwd_ret_5d", "bwd_ret_7d", "bwd_ret_14d", "bwd_ret_30d",
    "bwd_volume_ratio_1d", "bwd_volume_ratio_3d", "bwd_volume_ratio_5d",
    "bwd_volume_ratio_7d", "bwd_volume_ratio_14d", "bwd_volume_ratio_30d",
]

features = [c for c in features if c in df.columns]
cat_features = [
    c for c in features
    if c in [
        "source", "news_category", "sector", "subsector", "industry", "subindustry",
        "listing_board", "event_scope", "event_type", "event_side",
        "impact_channel", "materiality_label"
    ]
]

for part in [train, valid, test]:
    for c in features:
        if c in cat_features:
            part[c] = part[c].astype("string").fillna("UNKNOWN")
        else:
            part[c] = pd.to_numeric(part[c], errors="coerce").fillna(0)

train_pool = Pool(train[features], train[target], cat_features=cat_features)
valid_pool = Pool(valid[features], valid[target], cat_features=cat_features)
test_pool = Pool(test[features], test[target], cat_features=cat_features)

model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=80,
)

model.fit(train_pool, eval_set=valid_pool)

pred = model.predict_proba(test_pool)[:, 1]
auc = roc_auc_score(test[target], pred)
ap = average_precision_score(test[target], pred)

print("TEST AUC:", auc)
print("TEST AP :", ap)

test_pred_label = (pred >= 0.5).astype(int)
print(classification_report(test[target], test_pred_label, digits=4))

model.save_model(str(out / "news_risk_catboost.cbm"))

metrics = {
    "target": target,
    "features": features,
    "cat_features": cat_features,
    "train_rows": len(train),
    "valid_rows": len(valid),
    "test_rows": len(test),
    "test_auc": float(auc),
    "test_average_precision": float(ap),
}

(out / "news_risk_catboost_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved:", out)
PY